# EDA — k-space (FFT 2D) sobre los log-mel de `data/processed`

**Pregunta.** Si en vez de mirar el espectrograma log-mel `S(f, t)` miramos su transformada de Fourier 2D
`K(kv, ku)` (el "k-space", o *modulation spectrum* / dominio de ondulaciones espectro-temporales):

1. ¿Las clases (Normal, Fine crackle, Wheeze, Wheeze+Crackle) se **separan mejor**, incluso frente a las respiraciones normales?
2. ¿Se puede **filtrar ruido ambiental / respuesta del equipo** en ese dominio?
3. ¿Se **reduce el data shift** entre años (2022, 2023, 2024 → 2025)?

**Por qué podría funcionar.** Con `S = log-mel`, un efecto multiplicativo en el audio es *aditivo* en `S`, y cada estructura cae en una zona distinta de k-space:

| Estructura en el espectrograma | Dónde cae en k-space |
|---|---|
| Ruido ambiental estacionario / respuesta fija del estetoscopio o micrófono (offset por banda, constante en el tiempo) | columna **ku = 0** |
| Envolvente espectral suave (tilt, ganancia, timbre del torso) | **kv bajo** |
| Sibilancia (*wheeze*): línea horizontal estrecha y sostenida | **ku bajo, kv alto** (banda ancha en kv) |
| Crepitante (*crackle*): transitorio vertical de banda ancha | **kv ≈ 0, ku alto** |
| Desplazar el evento en tiempo o frecuencia | solo cambia la **fase**: `|K|` es invariante |

`ku` = modulación temporal (Hz, resolución de `FRAME_RATE = sr/hop = 31.25` fps → hasta 15.6 Hz);
`kv` = modulación espectral (ciclos por bin mel).

**Advertencias sobre estos datos (verificadas más abajo)**
- `train` (2022–2024) ya trae **SpecAugment** (rectángulos constantes en tiempo/frecuencia) y `test` (2025) no. Esos rectángulos dejan energía artificial en k-space y confunden "año" con "augmentation". Se detectan y se reparan (`MASK_POLICY`).
- `ConcatenationStage` pega eventos del mismo label y `PaddingStage` hace *tile*: hay empalmes y periodicidad que no son fisiológicos.
- Las columnas `patient_id`/`gender` de `index.csv` están desplazadas (bug de `parse_stem`): aquí el año se toma de `raw_stem`.
- Con `n_fft=512` a 4 kHz la ventana es de 128 ms, así que los crepitantes (< 20 ms) quedan muy suavizados en tiempo.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from scipy.fft import rfft2, irfft2, rfftfreq, fftfreq, fftshift
from scipy.signal.windows import hann, tukey

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.titlesize": 12, "axes.titleweight": "bold"})

In [ ]:
RANDOM_STATE = 42
PROC_ROOT = Path("../data/processed")

# Parametros del pipeline (workflow/config/pipeline.yaml -> log_mel)
SR, HOP, N_FFT, N_MELS, FMIN, FMAX = 4000, 128, 512, 64, 50, 2000
FRAME_RATE = SR / HOP                    # frames/s del espectrograma -> eje temporal de k-space en Hz

CLASSES = ["N", "FC", "W", "WC"]         # indice = clase numerica usada aqui (N = 0)
CLASS_NAMES = {"N": "Normal", "FC": "Fine crackle", "W": "Wheeze", "WC": "Wheeze+Crackle"}
CLASS_COLORS = dict(zip(CLASSES, sns.color_palette("Set2", 4)))
YEARS = [2022, 2023, 2024, 2025]
TRAIN_YEARS = [2022, 2023, 2024]         # todos con la misma augmentation -> comparables entre si

MAX_PER_YEAR_CLASS = 600                 # tope de muestras por (anio, clase); None = todas
# SpecAugment ya aplicado en train: "keep" | "drop" (descarta muestras enmascaradas) | "inpaint" (interpola la zona)
MASK_POLICY = "inpaint"
WINDOW = "tukey"                         # ventana 2D antes de la FFT: None | "hann" | "tukey"
PCA_DIM = 64                             # dimension tras PCA en los clasificadores de evaluacion
CHANNEL_SIGMA = 0.5                      # amplitud (en unidades z) del canal sintetico para la prueba de robustez

## 2. Carga de datos

Se toma una muestra balanceada por `(año, clase)`. El año sale de `raw_stem` (`2022`…`2025`);
2025 es el split `test` (el pipeline lo procesó con `augment=False`).

In [ ]:
frames = []
for split in ("train", "test"):
    d = pd.read_csv(PROC_ROOT / split / "index.csv")
    d["split"] = split
    frames.append(d)
index = pd.concat(frames, ignore_index=True)
index["year"] = index["raw_stem"].str[:4].astype(int)
index = index[index["label_str"].isin(CLASSES)]

parts = []
for (year, lab), g in index.groupby(["year", "label_str"]):
    n = len(g) if MAX_PER_YEAR_CLASS is None else min(len(g), MAX_PER_YEAR_CLASS)
    parts.append(g.sample(n, random_state=RANDOM_STATE))
meta = pd.concat(parts, ignore_index=True)

X = np.stack([np.load(PROC_ROOT / r.split / r.path) for r in tqdm(meta.itertuples(), total=len(meta), desc="features")]).astype(np.float32)
y = meta["label_str"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
year = meta["year"].to_numpy()
groups = meta["raw_stem"].to_numpy()          # grabacion de origen -> GroupKFold sin fuga

N_SAMPLES, _, T = X.shape
print("X:", X.shape, X.dtype)
pd.crosstab(meta["year"], meta["label_str"])[CLASSES]

### 2.1 SpecAugment residual

`SpecAugmentStage` rellena con `0.0` *antes* de normalizar, así que cada máscara queda como un frame (o un bin mel) **exactamente constante**.
Los detecto por desviación estándar nula y los reparo por interpolación lineal. Es una reparación aproximada
(el ruido aditivo / pitch / stretch de `WaveformAugmentationStage` en train no se puede deshacer).

In [ ]:
def detect_specaugment(X):
    t_mask = X.std(axis=1) < 1e-6          # (n, T): frame constante en frecuencia
    f_mask = X.std(axis=2) < 1e-6          # (n, F): bin mel constante en el tiempo
    return t_mask, f_mask


def _interp_fill(x, bad):
    # Rellena las posiciones `bad` (eje -1 de x) por interpolacion lineal entre vecinos validos
    L = bad.size
    if not bad.any() or bad.all():
        return x
    idx = np.arange(L)
    left = np.maximum.accumulate(np.where(~bad, idx, -1))
    right = np.minimum.accumulate(np.where(~bad, idx, L)[::-1])[::-1]
    j = np.where(bad)[0]
    l, r = left[j], right[j]
    l = np.where(l < 0, r, l)
    r = np.where(r >= L, l, r)
    w = np.where(r > l, (j - l) / np.maximum(r - l, 1), 0.0)
    out = x.copy()
    out[..., j] = x[..., l] * (1 - w) + x[..., r] * w
    return out


def inpaint_specaugment(X, t_mask, f_mask):
    Xo = X.copy()
    for i in np.where(t_mask.any(1) | f_mask.any(1))[0]:
        x = _interp_fill(Xo[i], t_mask[i])
        x = _interp_fill(x.T, f_mask[i]).T
        Xo[i] = (x - x.mean()) / (x.std() + 1e-8)       # el relleno de 0 sesgaba media y desviacion
    return Xo


t_mask, f_mask = detect_specaugment(X)
masked = t_mask.any(1) | f_mask.any(1)
print("Fraccion de muestras con SpecAugment detectado, por anio:")
print(pd.Series(masked).groupby(year).mean().round(3).to_string())

X_raw = X.copy()                       # copia con las mascaras originales (solo para la figura de abajo)
if MASK_POLICY == "inpaint":
    X = inpaint_specaugment(X, t_mask, f_mask)
elif MASK_POLICY == "drop":
    keep = ~masked
    X, y, year, groups, meta = X[keep], y[keep], year[keep], groups[keep], meta[keep].reset_index(drop=True)
    X_raw = X_raw[keep]
print(f"MASK_POLICY={MASK_POLICY!r}: X={X.shape}")

## 3. Definición del k-space

- Se aplica una ventana 2D (por defecto Tukey 0.25) antes de la FFT para limitar la fuga de espectro: sin ventana, los bordes del espectrograma
  dibujan una "cruz" sobre los ejes `ku = 0` y `kv = 0`, que son justo las zonas que nos interesan.
- Como `S` es real, se usa `rfft2`: se guarda `ku ∈ [0, 15.6 Hz]` y `kv ∈ [-0.5, 0.5)` ciclos/bin con signo (el signo de `kv` distingue barridos ascendentes de descendentes).
- Para la **invariancia exacta a un canal fijo** se resta la media temporal por banda *antes* de la ventana (`demean_time=True`).
  Restar después de ventanear (o anular el bin `ku = 0` del espectro ventaneado) deja fuga en `ku = 1, 2`.

In [ ]:
def make_window(shape, kind):
    if kind is None:
        return np.ones(shape, np.float32)
    f = {"hann": lambda n: hann(n, sym=False), "tukey": lambda n: tukey(n, 0.25)}[kind]
    return np.outer(f(shape[0]), f(shape[1])).astype(np.float32)


WIN = make_window((N_MELS, T), WINDOW)
KU_HZ = rfftfreq(T, d=1 / FRAME_RATE)             # (T//2+1,)  Hz
KV_CYC = fftfreq(N_MELS)                          # (F,)       ciclos/bin mel (orden FFT)
KV_IDX = np.abs(KV_CYC * N_MELS).astype(int)      # |kv| en ciclos por espectro, 0..32
KU_IDX = np.arange(len(KU_HZ))                    # indice de bin de ku
KU_STEP = KU_HZ[1]
K_EXTENT = [KU_HZ[0], KU_HZ[-1], fftshift(KV_CYC)[0], fftshift(KV_CYC)[-1]]


def hz_to_mel(f): return 2595 * np.log10(1 + f / 700)
def mel_to_hz(m): return 700 * (10 ** (m / 2595) - 1)
MEL_HZ = mel_to_hz(np.linspace(hz_to_mel(FMIN), hz_to_mel(FMAX), N_MELS + 2)[1:-1])


def kspace(X, demean_time=False):
    # FFT2 con ventana; sale (n, F, T//2+1) complejo
    Z = X - X.mean(-1, keepdims=True) if demean_time else X
    return rfft2(Z * WIN, axes=(-2, -1))


def klogmag(X, demean_time=False):
    return np.log1p(np.abs(kspace(X, demean_time)))


def show_spec(ax, x, title="", cmap="viridis", vmin=None, vmax=None):
    im = ax.imshow(x, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, extent=[0, x.shape[1] / FRAME_RATE, 0, x.shape[0]])
    ticks = [0, 16, 32, 48, 63]
    ax.set_yticks([t + 0.5 for t in ticks]); ax.set_yticklabels([f"{MEL_HZ[t]:.0f}" for t in ticks])
    ax.set_xlabel("t (s)"); ax.set_ylabel("f (Hz)"); ax.set_title(title)
    return im


def show_k(ax, K, title="", cmap="magma", vmin=None, vmax=None):
    im = ax.imshow(fftshift(K, axes=0), origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, extent=K_EXTENT)
    ax.set_xlabel("ku (Hz)  modulación temporal"); ax.set_ylabel("kv (ciclos/bin mel)"); ax.set_title(title)
    return im


print(f"resolucion ku: {KU_STEP:.3f} Hz/bin, max {KU_HZ[-1]:.1f} Hz | bins mel: {MEL_HZ[0]:.0f}-{MEL_HZ[-1]:.0f} Hz")

### 3.1 Efecto de SpecAugment en k-space

Misma muestra con su máscara original y reparada. Las máscaras rectangulares aparecen como "cruces" (sinc) de energía en k-space que no existen en la señal.

In [ ]:
i_mask = int(np.argmax(masked & (year == 2022) & (t_mask.sum(1) > 3)))
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for row, (xx, nm) in enumerate([(X_raw[i_mask], "con máscara original"), (X[i_mask], f"MASK_POLICY={MASK_POLICY!r}")]):
    show_spec(axes[row, 0], xx, f"Espectrograma — {nm}")
    show_k(axes[row, 1], klogmag(xx[None])[0], "log(1+|K|)", vmin=0, vmax=4.5)
plt.tight_layout(); plt.show()

## 4. Un ejemplo por clase

Fila de arriba: espectrograma (2025, sin augmentation). Abajo: `log(1+|K|)`.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for j, c in enumerate(CLASSES):
    i = rng.choice(np.where((year == 2025) & (y == j))[0])
    show_spec(axes[0, j], X[i], f"{CLASS_NAMES[c]}")
    show_k(axes[1, j], klogmag(X[i][None])[0], "log(1+|K|)", vmin=0, vmax=4.5)
plt.tight_layout(); plt.show()

## 5. ¿Dónde vive la información de clase y dónde el *domain shift*?

- **Mapa de clase**: F de ANOVA (una vía) de cada píxel de `log|K|` frente a la clase (todas las muestras).
- **Mapa de dominio**: F de ANOVA frente al **año**, *dentro de cada clase* (clases N, FC, W; solo 2022-2024, que comparten augmentation) y promediado; así el desbalance de clases entre años no cuenta como shift.
- Si el dominio se concentra en `ku = 0` y la clase no, un *notch* en esa columna quita shift sin perder señal.

In [ ]:
def anova_f(Z, g):
    # F de ANOVA de una via, por columna de Z (n, d), agrupando por g
    mu = Z.mean(0)
    k, n = len(np.unique(g)), len(Z)
    ssb = np.zeros(Z.shape[1]); ssw = np.zeros(Z.shape[1])
    for gi in np.unique(g):
        Zi = Z[g == gi]; mi = Zi.mean(0)
        ssb += len(Zi) * (mi - mu) ** 2
        ssw += ((Zi - mi) ** 2).sum(0)
    return (ssb / (k - 1)) / (ssw / (n - k) + 1e-12)


def domain_f(Z, y, year, min_n=5):
    # F frente al anio dentro de cada clase, promediado (el desbalance de clases entre anios no cuenta)
    out = []
    for c in np.unique(y):
        m = y == c
        yrs = [yy for yy in np.unique(year[m]) if (year[m] == yy).sum() >= min_n]
        mm = m & np.isin(year, yrs)
        if len(yrs) >= 2:
            out.append(anova_f(Z[mm], year[mm]))
    return np.mean(out, 0)


def top_share(F, q=0.05):
    f = np.sort(F.ravel())[::-1]
    return f[: max(1, int(q * f.size))].sum() / f.sum()


LM = klogmag(X)                                        # (n, F, Tk), ventana, sin demean
tr_cls = np.isin(year, TRAIN_YEARS) & (y < 3)          # N, FC, W en anios con la misma augmentation

F_cls_k = anova_f(LM.reshape(len(LM), -1), y).reshape(LM.shape[1:])
F_dom_k = domain_f(LM[tr_cls].reshape(tr_cls.sum(), -1), y[tr_cls], year[tr_cls]).reshape(LM.shape[1:])
F_cls_s = anova_f(X.reshape(len(X), -1), y).reshape(X.shape[1:])
F_dom_s = domain_f(X[tr_cls].reshape(tr_cls.sum(), -1), y[tr_cls], year[tr_cls]).reshape(X.shape[1:])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
show_spec(axes[0, 0], np.log10(F_cls_s), "Espectrograma: log10 F (clase)", cmap="viridis")
show_spec(axes[0, 1], np.log10(F_dom_s), "Espectrograma: log10 F (año | clase)", cmap="viridis")
show_k(axes[1, 0], np.log10(F_cls_k), "k-space: log10 F (clase)", cmap="viridis")
show_k(axes[1, 1], np.log10(F_dom_k), "k-space: log10 F (año | clase)", cmap="viridis")
plt.tight_layout(); plt.show()

print("Fraccion de la masa de F en el 5% de pixeles con mayor F (mas alto = mas concentrado):")
print(pd.DataFrame({"clase": [top_share(F_cls_s), top_share(F_cls_k)], "año|clase": [top_share(F_dom_s), top_share(F_dom_k)]},
                   index=["espectrograma", "k-space"]).round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(KU_HZ, F_cls_k.mean(0), "o-", label="clase")
ax.semilogy(KU_HZ, F_dom_k.mean(0), "s-", label="año | clase")
ax.set_xlabel("ku (Hz)"); ax.set_ylabel("F medio sobre kv"); ax.set_title("Perfil por ku"); ax.legend()
plt.tight_layout(); plt.show()

## 6. Mapa medio por clase y diferencia contra Normal

In [ ]:
mean_k = {c: LM[(y == j)].mean(0) for j, c in enumerate(CLASSES)}
vmax = max(m.max() for m in mean_k.values())
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for j, c in enumerate(CLASSES):
    show_k(axes[0, j], mean_k[c], f"{CLASS_NAMES[c]}: log(1+|K|) medio", vmin=0, vmax=vmax)
dmax = max(np.abs(mean_k[c] - mean_k["N"]).max() for c in CLASSES[1:])
axes[1, 0].axis("off")
for j, c in enumerate(CLASSES[1:], start=1):
    show_k(axes[1, j], mean_k[c] - mean_k["N"], f"{c} − N", cmap="RdBu_r", vmin=-dmax, vmax=dmax)
plt.tight_layout(); plt.show()

### 6.1 Regiones heurísticas

Reglas por física (ajustables): `STAT` = columna `ku = 0`; `WHEEZE` = `ku` bajo y `|kv|` alto; `CRACKLE` = `ku` alto y `|kv|` bajo; el resto es `OTRO`.
Es una hipótesis para leer los mapas, no un resultado: los mapas F de arriba son la evidencia.

In [ ]:
KU_SPLIT = 6      # bins de ku (6 -> 2 Hz): por debajo = sostenido (wheeze), por encima = transitorio (crackle)
KV_SPLIT = 6      # ciclos/espectro: por debajo = envolvente suave / transitorio de banda ancha, por encima = estructura fina en frecuencia

REGIONS = {
    "STAT (ku=0)":        (KU_IDX[None, :] == 0) & np.ones((N_MELS, 1), bool),
    "WHEEZE (ku bajo, kv alto)":  (KU_IDX[None, :] >= 1) & (KU_IDX[None, :] <= KU_SPLIT) & (KV_IDX[:, None] >= KV_SPLIT),
    "CRACKLE (ku alto, kv bajo)": (KU_IDX[None, :] > KU_SPLIT) & (KV_IDX[:, None] < KV_SPLIT),
}
other = ~np.any(list(REGIONS.values()), axis=0)
REGIONS["OTRO"] = other

fig, ax = plt.subplots(figsize=(6.5, 4.5))
lab = np.zeros((N_MELS, len(KU_IDX)))
for r, (nm, m) in enumerate(REGIONS.items()):
    lab[m] = r
im = ax.imshow(fftshift(lab, axes=0), origin="lower", aspect="auto", extent=K_EXTENT, cmap="Set3")
ax.set_xlabel("ku (Hz)"); ax.set_ylabel("kv (ciclos/bin mel)"); ax.set_title("Regiones heurísticas")
handles = [plt.Rectangle((0, 0), 1, 1, color=plt.cm.Set3(r / 11)) for r in range(len(REGIONS))]
ax.legend(handles, list(REGIONS), fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

tab = pd.DataFrame({CLASS_NAMES[c]: {nm: mean_k[c][m].mean() for nm, m in REGIONS.items()} for c in CLASSES})
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
sns.heatmap(tab, annot=True, fmt=".2f", cmap="viridis", ax=axes[0]); axes[0].set_title("log(1+|K|) medio por región")
sns.heatmap(tab.sub(tab["Normal"], axis=0), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1]); axes[1].set_title("Diferencia contra Normal")
plt.tight_layout(); plt.show()

## 7. Representaciones a comparar

Todas reciben `(n, 64, T)` z-scored y devuelven un vector por muestra. `fit` solo ve datos de entrenamiento.

| Nombre | Qué es |
|---|---|
| `espectrograma` | píxeles del log-mel (referencia) |
| `esp. − media temporal` | log-mel menos la media temporal por banda; **control**: es lo mismo que el notch `ku=0`, sin pasar por k-space |
| `k: log\|K\|` | magnitud en k-space, sin filtrar |
| `k: sin ku=0` | demean temporal + descarta la columna `ku=0` (canal / ruido estacionario) |
| `k: sin ku=0, sin kv<kv_lo` | además descarta la envolvente espectral suave |
| `k: pooled 8x8` | promedio por celdas de 8×6 bins: 64 números por muestra |
| `k: Fisher` | selecciona los píxeles con **alto F de clase y bajo F de año** (aprendido solo con los años de entrenamiento) |

In [ ]:
class Rep:
    def fit(self, X, y, year): return self
    def transform(self, X): raise NotImplementedError


class RawSpec(Rep):
    def transform(self, X): return X.reshape(len(X), -1)


class SpecCMN(Rep):
    def transform(self, X): return (X - X.mean(-1, keepdims=True)).reshape(len(X), -1)


def rect_mask(ku_lo=0, ku_hi=None, kv_lo=0, kv_hi=None):
    m = KU_IDX[None, :] >= ku_lo
    if ku_hi is not None: m = m & (KU_IDX[None, :] <= ku_hi)
    m = m & (KV_IDX[:, None] >= kv_lo)
    if kv_hi is not None: m = m & (KV_IDX[:, None] <= kv_hi)
    return m


class KLogMag(Rep):
    # log|K| restringido a un rectangulo (ku_lo..ku_hi, |kv| kv_lo..kv_hi). ku_lo>=1 implica demean temporal exacto
    def __init__(self, ku_lo=0, ku_hi=None, kv_lo=0, kv_hi=None):
        self.demean = ku_lo >= 1
        self.mask = rect_mask(ku_lo, ku_hi, kv_lo, kv_hi)

    def transform(self, X): return klogmag(X, self.demean)[:, self.mask]


class KPooled(Rep):
    def __init__(self, nv=8, nu=8, demean=True):
        self.nv, self.nu, self.demean = nv, nu, demean

    def transform(self, X):
        L = klogmag(X, self.demean)[:, :, 1:] if self.demean else klogmag(X)    # sin ku=0 si hay demean
        n, F, Tk = L.shape
        Tk = (Tk // self.nu) * self.nu
        return L[:, :, :Tk].reshape(n, self.nv, F // self.nv, self.nu, Tk // self.nu).mean((2, 4)).reshape(n, -1)


class KFisher(Rep):
    # Mascara en k-space: alto F de clase y bajo F de anio (dentro de clase). Aprendida solo con los datos de fit
    def __init__(self, keep=0.15, lam=1.0, demean=True):
        self.keep, self.lam, self.demean = keep, lam, demean

    def fit(self, X, y, year):
        L = klogmag(X, self.demean)
        Z = L.reshape(len(L), -1)
        score = np.log(anova_f(Z, y) + 1e-9) - self.lam * np.log1p(domain_f(Z, y, year))
        score = score.reshape(L.shape[1:])
        if self.demean: score[:, 0] = -np.inf
        thr = np.quantile(score[np.isfinite(score)], 1 - self.keep)
        self.mask_ = score >= thr
        self.score_ = score
        return self

    def transform(self, X): return klogmag(X, self.demean)[:, self.mask_]

### 7.1 Máscara de Fisher y filtros de ejemplo (reconstrucción)

Para ver qué hace cada filtro sobre el espectrograma se reconstruye con `irfft2` (sin ventana, que no es invertible).
La máscara de Fisher se aprende con 2022-2024 y se muestra sobre muestras de 2025.

In [ ]:
fisher_demo = KFisher(keep=0.15, lam=1.0).fit(X[np.isin(year, TRAIN_YEARS)], y[np.isin(year, TRAIN_YEARS)], year[np.isin(year, TRAIN_YEARS)])


def kfilter(x, mask):
    # x (F,T) -> filtra en k-space (sin ventana) con mask (F, T//2+1) booleana/float y vuelve al espectrograma
    return irfft2(rfft2(x) * mask, s=x.shape)


FILTERS = {
    "original": None,
    "sin ku=0 (canal/ruido estacionario)": rect_mask(ku_lo=1),
    "sin ku=0 y sin kv<4 (envolvente)": rect_mask(ku_lo=1, kv_lo=4),
    "máscara Fisher (clase vs año)": fisher_demo.mask_,
}

fig, axes = plt.subplots(len(CLASSES), len(FILTERS), figsize=(4 * len(FILTERS), 2.7 * len(CLASSES)))
rng = np.random.default_rng(1)
for r, c in enumerate(CLASSES):
    i = rng.choice(np.where((year == 2025) & (y == r))[0])
    for col, (nm, m) in enumerate(FILTERS.items()):
        xx = X[i] if m is None else kfilter(X[i], m)
        lim = np.percentile(np.abs(xx), 99)                 # escala propia: lo filtrado tiene mucha menos energia
        show_spec(axes[r, col], xx, CLASS_NAMES[c], cmap="RdBu_r", vmin=-lim, vmax=lim)
[axes[0, k].set_title(n) for k, n in enumerate(FILTERS)]
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show_k(axes[0], fisher_demo.score_, "score = log F(clase) − λ·log(1+F(año))", cmap="viridis")
show_k(axes[1], fisher_demo.mask_.astype(float), f"máscara (keep={fisher_demo.keep:.0%})", cmap="gray_r")
plt.tight_layout(); plt.show()

## 8. Evaluación: separabilidad, robustez a canal y *data shift*

Clasificador fijo y simple para todas las representaciones: `StandardScaler → PCA(64) → LogisticRegression(class_weight="balanced")`.
Si la representación es mejor, que lo sea con un modelo lineal.

1. **Leave-one-year-out (LOYO)** sobre 2022-2025: macro-F1, y AUC Normal vs anormal (`1 − P(N)`). El año 2025 es el único sin augmentation (`F1` de 2023 es ruidoso: casi no hay WC).
2. **Robustez a canal**: el modelo entrenado en datos limpios se evalúa en el año retenido con un **canal sintético** (offset suave en frecuencia, 1-3 ciclos por espectro, amplitud `CHANNEL_SIGMA` en unidades z; se vuelve a normalizar). Mide cuánto "aguanta" cada representación un cambio de equipo.
3. **Domain accuracy**: ¿qué tan bien se adivina el año (2022 / 2023 / 2024) a partir de la representación? Celdas `(año, clase)` balanceadas en N, FC, W y `GroupKFold` por grabación. **Menor es mejor** (azar = 0.33).

Sobre la 3.ª: no se incluye 2025 porque difiere de 2022-2024 también en augmentation, y la comparación mediría eso.

In [ ]:
def channel_perturb(X, sigma, seed=0):
    # offset aditivo suave en frecuencia (canal fijo) + re-normalizacion por muestra, como hace NormalizationStage
    rng = np.random.default_rng(seed)
    f = np.linspace(0, 1, X.shape[1])[None, :]
    c = np.zeros((len(X), X.shape[1]), np.float32)
    for q in (1, 2, 3):
        c += rng.normal(0, sigma / np.sqrt(3), (len(X), 1)) * np.cos(2 * np.pi * q * f + rng.uniform(0, 2 * np.pi, (len(X), 1)))
    Xp = X + c[:, :, None]
    return (Xp - Xp.mean((1, 2), keepdims=True)) / (Xp.std((1, 2), keepdims=True) + 1e-8)


X_chan = channel_perturb(X, CHANNEL_SIGMA)


def make_clf():
    return make_pipeline(StandardScaler(), PCA(PCA_DIM, svd_solver="randomized", random_state=RANDOM_STATE),
                         LogisticRegression(C=1.0, class_weight="balanced", max_iter=500))


def eval_loyo(make_rep):
    rows = []
    for hold in YEARS:
        tr, te = year != hold, year == hold
        rep = make_rep().fit(X[tr], y[tr], year[tr])
        clf = make_clf().fit(rep.transform(X[tr]), y[tr])
        for cond, Xte in (("limpio", X[te]), ("canal", X_chan[te])):
            P = clf.predict_proba(rep.transform(Xte))
            rows.append(dict(hold=hold, cond=cond,
                             macro_f1=f1_score(y[te], P.argmax(1), average="macro", labels=range(4), zero_division=0),
                             bal_acc=balanced_accuracy_score(y[te], P.argmax(1)),
                             auc_abn=roc_auc_score(y[te] > 0, 1 - P[:, 0])))
    return pd.DataFrame(rows)


# celdas (anio, clase) balanceadas para el clasificador de dominio
_cell = min(((year == yy) & (y == c)).sum() for yy in TRAIN_YEARS for c in (0, 1, 2))
_cell = min(_cell, 300)
_rng = np.random.default_rng(RANDOM_STATE)
DOM_IDX = np.concatenate([_rng.choice(np.where((year == yy) & (y == c))[0], _cell, replace=False)
                          for yy in TRAIN_YEARS for c in (0, 1, 2)])
print(f"dominio: {_cell} muestras por (año, clase) -> {len(DOM_IDX)} en total")


def eval_domain(make_rep, n_splits=4):
    Xd, yd, yrd, gd = X[DOM_IDX], y[DOM_IDX], year[DOM_IDX], groups[DOM_IDX]
    accs, aucs = [], []
    for tr, te in GroupKFold(n_splits).split(Xd, yrd, gd):
        rep = make_rep().fit(Xd[tr], yd[tr], yrd[tr])
        clf = make_clf().fit(rep.transform(Xd[tr]), yrd[tr])
        P = clf.predict_proba(rep.transform(Xd[te]))
        accs.append(balanced_accuracy_score(yrd[te], clf.classes_[P.argmax(1)]))
        aucs.append(roc_auc_score(yrd[te], P, multi_class="ovr", average="macro", labels=clf.classes_))
    return float(np.mean(accs)), float(np.mean(aucs))


RESULTS, PER_YEAR = {}, {}


def run_config(name, make_rep):
    r = eval_loyo(make_rep)
    dom_acc, dom_auc = eval_domain(make_rep)
    PER_YEAR[name] = r
    g = r.groupby("cond")[["macro_f1", "auc_abn"]].mean()
    RESULTS[name] = {"F1 limpio": g.loc["limpio", "macro_f1"], "F1 canal": g.loc["canal", "macro_f1"],
                     "AUC N-vs-anormal": g.loc["limpio", "auc_abn"], "AUC (canal)": g.loc["canal", "auc_abn"],
                     "domain acc": dom_acc, "domain AUC": dom_auc}
    return RESULTS[name]

In [ ]:
MAIN = {
    "espectrograma":                  lambda: RawSpec(),
    "esp. − media temporal":          lambda: SpecCMN(),
    "k: log|K|":                      lambda: KLogMag(),
    "k: sin ku=0":                    lambda: KLogMag(ku_lo=1),
    "k: sin ku=0, kv>=4":             lambda: KLogMag(ku_lo=1, kv_lo=4),
    "k: pooled 8x8":                  lambda: KPooled(),
    "k: Fisher (keep 15%, λ=1)":      lambda: KFisher(keep=0.15, lam=1.0),
}
for name, mk in tqdm(MAIN.items(), desc="representaciones"):
    run_config(name, mk)

res = pd.DataFrame(RESULTS).T
res.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
pos = np.arange(len(res)); w = 0.38
axes[0].bar(pos - w / 2, res["F1 limpio"], w, label="limpio"); axes[0].bar(pos + w / 2, res["F1 canal"], w, label="con canal sintético")
axes[0].set_title("Macro-F1 (LOYO, media de 4 años)")
axes[1].bar(pos - w / 2, res["AUC N-vs-anormal"], w, label="limpio"); axes[1].bar(pos + w / 2, res["AUC (canal)"], w, label="con canal sintético")
axes[1].set_ylim(0.5, None); axes[1].set_title("AUC Normal vs anormal (LOYO)")
axes[2].bar(pos, res["domain acc"], color="tab:red"); axes[2].axhline(1 / 3, color="k", ls="--", lw=1)
axes[2].set_title("¿Se adivina el año? (menor = más invariante)")
for a in axes:
    a.set_xticks(pos); a.set_xticklabels(res.index, rotation=35, ha="right")
axes[0].legend(); plt.tight_layout(); plt.show()

per = pd.concat({k: v for k, v in PER_YEAR.items()}, names=["rep"]).reset_index(level=0)
per[per.cond == "limpio"].pivot(index="rep", columns="hold", values="macro_f1").loc[list(MAIN)].round(3)

### 8.1 Sensibilidad a los cortes y compromiso F1 / invariancia

Barrido de `ku_lo`, `kv_lo` y de la penalización de dominio `λ`. En el gráfico, **arriba y a la izquierda** es lo deseable
(buen F1, año difícil de adivinar). La línea punteada del *domain acc* es el azar.

In [ ]:
SWEEP = {}
for ku in (2, 3):                          # ku>=1 ya esta en MAIN ("k: sin ku=0")
    SWEEP[f"k: ku>={ku}"] = (lambda ku=ku: KLogMag(ku_lo=ku))
for kv in (2, 8, 12):
    SWEEP[f"k: ku>=1, kv>={kv}"] = (lambda kv=kv: KLogMag(ku_lo=1, kv_lo=kv))
SWEEP["k: ku>=1, kv<=6 (transitorios)"] = lambda: KLogMag(ku_lo=1, kv_hi=6)
for lam in (0.0, 3.0):
    SWEEP[f"k: Fisher (λ={lam:g})"] = (lambda lam=lam: KFisher(keep=0.15, lam=lam))
SWEEP["k: Fisher (keep 5%, λ=1)"] = lambda: KFisher(keep=0.05, lam=1.0)

for name, mk in tqdm(SWEEP.items(), desc="barrido"):
    run_config(name, mk)

res = pd.DataFrame(RESULTS).T
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, ttl in zip(axes, ["F1 limpio", "F1 canal"], ["Macro-F1 limpio", "Macro-F1 con canal sintético"]):
    ax.scatter(res["domain acc"], res[col], c=["tab:red" if n in ("espectrograma", "esp. − media temporal") else "tab:blue" for n in res.index])
    for n, r_ in res.iterrows():
        ax.annotate(n, (r_["domain acc"], r_[col]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.axvline(1 / 3, color="k", ls="--", lw=1)
    ax.set_xlabel("domain acc (menor = más invariante al año)"); ax.set_ylabel(ttl)
plt.tight_layout(); plt.show()
res.sort_values("F1 limpio", ascending=False).round(3)

## 9. Vista 2D (t-SNE)

PCA(30) → t-SNE sobre la misma representación. La máscara de Fisher se ajusta **solo con 2022-2024**; la fila de arriba pinta únicamente puntos de 2025 (retenido) por clase,
la de abajo todos los puntos por año. Muestra balanceada por (año, clase).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sub = np.concatenate([rng.choice(np.where((year == yy) & (y == c))[0], min(150, ((year == yy) & (y == c)).sum()), replace=False)
                      for yy in YEARS for c in range(4)])
trm = np.isin(year, TRAIN_YEARS)
reps_tsne = {"espectrograma": RawSpec().fit(X[trm], y[trm], year[trm]),
             "k: sin ku=0": KLogMag(ku_lo=1).fit(X[trm], y[trm], year[trm]),
             "k: Fisher": KFisher().fit(X[trm], y[trm], year[trm])}

fig, axes = plt.subplots(2, len(reps_tsne), figsize=(5.2 * len(reps_tsne), 9))
for j, (nm, rep) in enumerate(reps_tsne.items()):
    F_ = StandardScaler().fit_transform(rep.transform(X[sub]))
    Z = TSNE(2, perplexity=30, init="pca", random_state=RANDOM_STATE).fit_transform(PCA(30, random_state=RANDOM_STATE).fit_transform(F_))
    is25 = year[sub] == 2025
    for k, c in enumerate(CLASSES):
        m = is25 & (y[sub] == k)
        axes[0, j].scatter(Z[m, 0], Z[m, 1], s=10, color=CLASS_COLORS[c], label=CLASS_NAMES[c])
    axes[0, j].set_title(f"{nm}\n2025 por clase")
    for yy in YEARS:
        m = year[sub] == yy
        axes[1, j].scatter(Z[m, 0], Z[m, 1], s=6, alpha=0.6, label=str(yy))
    axes[1, j].set_title("todos los años")
axes[0, 0].legend(fontsize=8); axes[1, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. Controles y limitaciones

Dos comprobaciones para no leer de más los resultados:
1. **Duración como atajo.** `duration_s` (duración del primer evento) difiere mucho entre clases y la concatenación/tile deja periodicidades que k-space puede capturar. Si un clasificador con *solo* la duración ya rinde parecido, parte de la "separabilidad" no es acústica.
2. **Qué falta.** Entrada = log-mel ya normalizado: no se puede simular ruido en el dominio lineal ni probar SNR reales. El canal sintético es solo un offset en dB constante en el tiempo, que `ku=0` cancela *por construcción*; una robustez real exigiría audios de otros dispositivos. Antes de adoptar una representación hay que repetir esto sobre audios sin SpecAugment ni tile.

In [ ]:
dur = meta["duration_s"].to_numpy()[:, None]
rows = []
for hold in YEARS:
    tr, te = year != hold, year == hold
    clf = LogisticRegression(class_weight="balanced", max_iter=500).fit(np.log(dur[tr]), y[tr])
    rows.append(f1_score(y[te], clf.predict(np.log(dur[te])), average="macro", labels=range(4), zero_division=0))
print(f"Macro-F1 LOYO usando SOLO log(duration_s): {np.mean(rows):.3f}  (por año: {np.round(rows, 3).tolist()})")
print("Ver tabla de la sección 8 para comparar; azar con 4 clases balanceadas ~0.25.")

## 11. Hallazgos de la corrida inicial

Corrida con `RANDOM_STATE=42`, `MAX_PER_YEAR_CLASS=600`, `MASK_POLICY="inpaint"`, ventana Tukey. Los números exactos cambian con la muestra; lo que importa es el orden y los signos.

1. **Dónde vive cada cosa.** El F de *año dentro de clase* se concentra en `ku=0` y `|kv|` bajo (canal / envolvente), como predice la tabla del inicio. El F de *clase* es un bloque ancho en `ku≈1–6 Hz`, `|kv|≈0.1`. Ambos están separados, así que filtrar el primero sin tocar el segundo es posible.
2. **Separabilidad.** `log|K|` da macro-F1 ≈ 0.53 y AUC Normal-vs-anormal ≈ 0.84, frente a ≈ 0.36 / 0.71 del espectrograma plano. Ojo: la referencia es un modelo lineal sobre píxeles, que no es invariante a desplazamientos; `|K|` sí lo es, y buena parte de la ganancia viene de ahí. No está comparado contra una CNN. Control: usar solo `log(duration_s)` da F1 ≈ 0.25 (azar), así que la duración del primer evento no explica la ganancia.
3. **El notch `ku=0` no es lo que mejora la separación.** Quitar `ku=0` deja el F1 igual (0.53 → 0.54) y baja un poco el *domain acc* (0.445 → 0.42). El control `esp. − media temporal`, que hace lo mismo sin pasar por k-space, es el peor (0.29): lo que ayuda es usar la **magnitud** en k-space.
4. **Compromiso F1 / invariancia.** Descartar `|kv|<2` además del notch (`k: ku>=1, kv>=2`) baja el *domain acc* a ≈ 0.37 (azar 0.33) con F1 ≈ 0.52: es el mejor punto del barrido. Cortes más agresivos (`kv>=8`, Fisher con `keep=5%`) acercan el dominio al azar pero hunden el F1 (≤ 0.43).
5. **Canal sintético.** Las variantes en k-space no se mueven (F1 0.53 → 0.53); el espectrograma cae (0.36 → 0.31, AUC 0.71 → 0.64). Es esperable y no prueba robustez real: el canal es constante en el tiempo, justo lo que `ku=0` cancela por construcción.
6. **Lo que NO se resolvió.** Con 2025 retenido el F1 es 0.34–0.41 frente a 0.5–0.6 en los otros años, y en el t-SNE 2025 queda aparte de 2022–24 incluso en k-space. 2025 es el único año sin SpecAugment/ruido aditivo, así que no se puede separar *shift real* de *diferencia de augmentation*.
7. **Para revisar.** En la sección 6.1 `Wheeze` sube en **todas** las regiones (no solo en la de sibilancia); puede deberse a empalmes de concatenación (sus eventos son los más cortos) y no a acústica. Falta probarlo con eventos sin concatenar.